In [1]:
import cmlreaders as cml
from cmlreaders import CMLReader, get_data_index
import pandas as pd 
import numpy as np
import os 


# Experiment 1

In [2]:
data = cml.get_data_index(kind = 'ltp'); data = data[data['experiment']=='VFFR']
sess_evs_list = []
for i, row in data.iterrows():
    try:
        reader = CMLReader(subject=row['subject'], session=row['session'], experiment=row['experiment'])
        sess_evs = reader.load('task_events')
        sess_evs_list.append(sess_evs)
    except:
        print(row)

all_events                                                        NaN
experiment                                                       VFFR
import_type                                                     build
math_events                                                       NaN
original_session                                                    0
session                                                             0
subject                                                        LTP414
subject_alias                                                  LTP414
task_events         protocols/ltp/subjects/LTP414/experiments/VFFR...
Name: 6653, dtype: object
all_events                                                        NaN
experiment                                                       VFFR
import_type                                                     build
math_events                                                       NaN
original_session                                                

In [3]:
exp1_all_evs = pd.concat(sess_evs_list)

In [4]:
exp1_all_evs.to_csv('dataframes/exp1_all_evs.csv', index=False)

In [5]:
exp1_ffr_evs = exp1_all_evs.query('type == ["WORD", "FFR_REC_WORD"]')

In [6]:
# assign item number in the order of presentation
item_col = 'item_name'
item_num_col = 'item_num'
item_num_df = exp1_ffr_evs.query("type=='WORD'").drop_duplicates(subset=item_col, ignore_index=True
                                    )[item_col].reset_index().rename(columns={'index': item_num_col})
item_num_df[item_num_col] = item_num_df[item_num_col] + 1
events_new = exp1_ffr_evs.merge(item_num_df, on=item_col, suffixes=('', '_new'), 
                          how='left', sort=False)#.sort_values('mstime')
events_new.fillna({'item_num_new': -999}, inplace=True)

In [7]:
exp1_ffr_evs = events_new

In [8]:
# number of sessions per subject
n_sess_df = exp1_ffr_evs.groupby('subject', as_index=False).agg({'session': 'nunique'})

In [9]:
# subjects who contributed fewer than 7 sessions
few_sess_subs_7 = n_sess_df.query('session < 7').subject.values.tolist()

In [10]:
# only include subjects who scored above .7 on every session
prop_correct_df = exp1_ffr_evs.query('type == "WORD"').groupby(['subject', 'session']).agg({'correct': 'sum'}).reset_index()
prop_correct_df['prop_correct'] = prop_correct_df['correct'] / (24 * 24)
low_prop_correct_subs = prop_correct_df.query('prop_correct < .7').subject.values.tolist()

In [11]:
exclude_subs = few_sess_subs_7 + low_prop_correct_subs 
exp1_ffr_evs_KateEtal22 = exp1_ffr_evs.query('subject != @exclude_subs and session >= 4 and session <= 10')

In [12]:
exp1_ffr_evs_KateEtal22.to_csv('dataframes/KateEtal22_filter_exp1_ffr_evs.csv', index=False)

In [2]:
from gensim.models import KeyedVectors
word2vec_vectors = KeyedVectors.load_word2vec_format("/scratch/rafla/GoogleNews-vectors-negative300.bin", binary=True)

/home1/rafla/.local/lib/python3.7/site-packages/gensim/similarities/__init__.py:15: UserWarning: The gensim.similarities.levenshtein submodule is disabled, because the optional Levenshtein package <https://pypi.org/project/python-Levenshtein/> is unavailable. Install Levenhstein (e.g. `pip install python-Levenshtein`) to suppress this warning.
  warnings.warn(msg)


In [14]:
def word_similarity(df, col1, col2, keyed_vector=None):
    try:
        return keyed_vector.similarity(df[col1].lower(), df[col2].lower())
    except:
        return np.nan

In [15]:
# get all pairs of items
items = item_num_df.item_name.values
sem_sim_df = pd.MultiIndex.from_product([items, items], names=['item_1', 'item_2']).to_frame(index=False)

In [16]:
# compute similarity of all pairs
sem_sim_df['similarity'] = sem_sim_df.apply(word_similarity, 
               axis=1, col1='item_1', 
               col2='item_2', 
               keyed_vector=word2vec_vectors)

In [17]:
sem_sim_num_df = sem_sim_df.merge(
    item_num_df, left_on='item_1', right_on=item_col).merge(
    item_num_df, left_on='item_2', right_on=item_col, suffixes=('_1', '_2')).drop(columns=['item_1', 'item_2'])

In [18]:
sem_sim_num_df.to_csv('dataframes/exp1_sem_sim_num_df.csv', index=False)

# Experiment 2

In [19]:
data = cml.get_data_index(kind = 'ltp'); data = data[data['experiment']=='ltpRepFR']
sess_evs_list = []
for i, row in data.iterrows():
    try:
        reader = CMLReader(subject=row['subject'], session=row['session'], experiment=row['experiment'])
        sess_evs = reader.load('task_events')
        sess_evs_list.append(sess_evs)
    except:
        print(row)

In [26]:
data = cml.get_data_index(); 

In [31]:
repFR1 = data.query('experiment=="RepFR1"')

In [37]:
repFR1.subject.unique()

array(['R1204T', 'R1501J', 'R1514E', 'R1516E', 'R1528E', 'R1531T',
       'R1534D', 'R1547D', 'R1556J', 'R1564J', 'R1566D', 'R1568E',
       'R1579T', 'R1582E', 'R1584J', 'R1586T', 'R1587J', 'R1589T',
       'R1590T', 'R1593D', 'R1594E', 'R1596T', 'R1603T', 'R1604J',
       'R1610D', 'R1611T', 'R1612E', 'R1613T', 'R1615T', 'R1618J',
       'R1619T', 'R1620J', 'R1621E', 'R1622T', 'R1624E', 'R1625T',
       'R1627T', 'R1628E', 'R1630E', 'R1632E', 'R1633J', 'R1635T',
       'R1638E', 'R1642J', 'R1644T', 'R1650E', 'R1651J', 'R1653J',
       'R1654J', 'R1670J', 'R1679J', 'R1681J', 'R1690E', 'R1692T'],
      dtype=object)

In [38]:
reader = CMLReader('R1501J', session=0, experiment='RepFR1')

In [40]:
reader.load('task_events').query('type=="WORD"').list.unique()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25])

In [19]:
data = data[data['experiment']=='RepFR1']

In [23]:
data.experiment.unique()

array(['ltpFR', 'ltpFR2', 'VFFR', 'ltpRepFR', 'NiclsCourierClosedLoop',
       'NiclsCourierReadOnly', 'ltpDelayRepFRReadOnly',
       'CourierReinstate1', 'ltpDBOY1', 'prelim'], dtype=object)

In [20]:
exp2_all_evs = pd.concat(sess_evs_list)

In [21]:
exp2_all_evs.to_csv('dataframes/exp2_all_evs.csv', index=False)

In [5]:
pd.read_csv('dataframes/exp2_all_evs.csv').type.unique()

/usr/global/miniconda/py310_23.1.0-1/envs/workshop/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (7,21,24,25) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


array(['SESS_START', 'TRIAL_START', 'COUNTDOWN', 'WORD', 'WORD_OFF',
       'REC_START', 'REC_WORD', 'REC_WORD_VV', 'REC_END', 'BREAK_START',
       'BREAK_STOP', 'SESS_END'], dtype=object)

In [2]:
all_evs = pd.read_csv('dataframes/exp2_all_evs.csv')

/usr/global/miniconda/py310_23.1.0-1/envs/workshop/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (7,21,24,25) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [5]:
items = all_evs.query('type=="WORD"').item_name.unique()
len(items)

312

In [10]:
for item in items:
    print(item)
    item_df = all_evs.query("type=='WORD' and item_name==@item")
    print(len(item_df))
    print(item_df.list.to_list())
    print(len(item_df.list.unique()))

STOOL
145
[0, 0, 5, 5, 5, 20, 20, 15, 15, 15, 5, 5, 5, 15, 9, 9, 16, 16, 16, 21, 21, 3, 3, 3, 24, 24, 24, 15, 15, 15, 20, 20, 20, 11, 11, 1, 15, 15, 1, 22, 22, 22, 10, 10, 10, 19, 1, 10, 10, 10, 2, 2, 2, 9, 9, 9, 5, 3, 3, 3, 11, 11, 15, 15, 17, 17, 8, 8, 8, 7, 7, 6, 6, 6, 5, 5, 19, 19, 19, 15, 15, 15, 12, 12, 7, 7, 10, 4, 1, 1, 1, 23, 23, 23, 23, 23, 8, 8, 1, 1, 1, 21, 21, 21, 13, 18, 18, 18, 3, 3, 4, 10, 10, 10, 10, 18, 18, 18, 18, 18, 9, 9, 20, 4, 4, 4, 24, 16, 25, 25, 18, 25, 25, 6, 17, 17, 17, 7, 1, 1, 1, 7, 7, 7, 5]
25
THRONE
161
[0, 0, 17, 17, 17, 8, 8, 8, 8, 8, 7, 7, 7, 13, 13, 21, 21, 21, 25, 14, 14, 14, 22, 22, 17, 17, 17, 25, 25, 25, 2, 2, 2, 2, 0, 0, 0, 25, 25, 2, 2, 21, 21, 10, 10, 10, 9, 13, 13, 17, 17, 17, 10, 10, 10, 8, 8, 8, 5, 5, 5, 14, 22, 22, 22, 3, 3, 3, 12, 0, 0, 0, 25, 25, 0, 0, 25, 25, 25, 6, 22, 25, 25, 25, 18, 18, 18, 1, 1, 1, 22, 22, 22, 23, 23, 13, 13, 13, 13, 14, 21, 21, 21, 11, 11, 11, 15, 15, 15, 22, 22, 22, 5, 16, 16, 4, 4, 2, 2, 2, 5, 5, 5, 17, 17, 0, 11

In [22]:
exp2_ffr_evs = exp2_all_evs.query('type == ["WORD", "REC_WORD", "FFR_REC_WORD"]')

In [23]:
item_col = 'item_name'
item_num_col = 'item_num'
item_num_df = exp2_ffr_evs.query("type=='WORD'").drop_duplicates(subset=item_col, ignore_index=True
                                    )[item_col].reset_index().rename(columns={'index': item_num_col})
item_num_df[item_num_col] = item_num_df[item_num_col] + 1
events_new = exp2_ffr_evs.merge(item_num_df, on=item_col, suffixes=('', '_new'), 
                          how='left', sort=False)#.sort_values('mstime')
events_new.fillna({'item_num_new': -999}, inplace=True)

In [24]:
exp2_ffr_evs = events_new

In [25]:
exp2_ffr_evs.to_csv('dataframes/exp2_ffr_evs.csv', index=False)

In [26]:
def word_similarity(df, col1, col2, keyed_vector=None):
    try:
        return keyed_vector.similarity(df[col1].lower(), df[col2].lower())
    except:
        return np.nan

In [27]:
# get all pairs of items
items = item_num_df.item_name.values
sem_sim_df = pd.MultiIndex.from_product([items, items], names=['item_1', 'item_2']).to_frame(index=False)

In [28]:
# compute similarity of all pairs
sem_sim_df['similarity'] = sem_sim_df.apply(word_similarity, 
               axis=1, col1='item_1', 
               col2='item_2', 
               keyed_vector=word2vec_vectors)

In [29]:
sem_sim_num_df = sem_sim_df.merge(
    item_num_df, left_on='item_1', right_on=item_col).merge(
    item_num_df, left_on='item_2', right_on=item_col, suffixes=('_1', '_2')).drop(columns=['item_1', 'item_2'])

In [30]:
sem_sim_num_df.to_csv('dataframes/exp2_sem_sim_num_df.csv', index=False)